# Self-Improving Agents: Reflection and Reflexion with Persistent Memory

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **Reflection** pattern and how message-role manipulation drives iterative refinement.
2. Build a **generate-critique-refine** loop using LangGraph's `StateGraph`.
3. Understand how **Reflexion** extends Reflection with persistent vector memory (FAISS).
4. Implement a **Reflexion agent** that learns from past mistakes across sessions.
5. Choose the right pattern for different use cases.

---

## How This Differs from the Evaluator-Optimiser Pattern

In Notebook 04 (`langchain-patterns/04_Evaluator_Optimiser`), a **separate Evaluator LLM** grades output against fixed criteria using Pydantic structured output. That pattern is about compliance checking with two distinct LLM roles.

**Reflection** is fundamentally different: a single LLM plays both writer and critic through **message-role manipulation**. The critique is sent back as a `HumanMessage`, tricking the generator into treating its own work as "someone else's submission to improve." This is a psychological trick on the LLM, not a structured evaluation.

**Reflexion** goes further by adding **persistent vector memory** (FAISS) so the agent remembers its past mistakes and avoids repeating them across sessions.

---

## Key Concepts

| Concept | Description |
|---|---|
| **Reflection** | Generate-critique-refine loop using message-role swapping |
| **Role Manipulation** | Converting `AIMessage` to `HumanMessage` so the LLM critiques its own output as if reviewing someone else's work |
| **Reflexion** | Reflection + persistent FAISS memory + web search + structured self-critique |
| **Vector Memory** | FAISS store that persists reflections across sessions for continuous improvement |
| **Structured Self-Critique** | Pydantic schemas (`AnswerQuestion`, `Reflection`) for organized feedback |

## Step 1: Setup and Azure OpenAI Configuration

We initialize a single LLM that will play both the **generator** and **critic** roles. Unlike the Evaluator-Optimiser pattern (which uses two LLMs at different temperatures), Reflection uses one LLM with different system prompts.

In [1]:
# Install required packages (uncomment if needed)
# !pip install langgraph langchain-openai langchain-core python-dotenv faiss-cpu langchain-community

import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

# ── Initialize the LLM ────────────────────────────────────────────────────────
llm = AzureChatOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1-mini")
)

print("Azure OpenAI model initialized.")
print(f"   Model: {os.getenv('AZURE_OPENAI_MODEL', 'gpt-4.1-mini')}")
print("   One LLM, two roles: generator and critic.")

Azure OpenAI model initialized.
   Model: gpt-4.1-mini
   One LLM, two roles: generator and critic.


## Step 2: The Reflection Pattern

Reflection is a **generate-critique-refine** loop. The key insight is **message-role manipulation**: when the generator produces an `AIMessage`, the reflection node converts it to a `HumanMessage` so the generator sees it as "student work to improve" rather than "my own output."

### Architecture

```
User Request (HumanMessage)
        |
        v
  [Generator Node]  --->  Draft essay (AIMessage)
        ^                       |
        |                       v
        |              [Reflection Node]
        |                 - Swaps AI messages to Human messages
        |                 - Critic reviews "the student's essay"
        |                 - Returns critique as HumanMessage
        |                       |
        +-----------------------+
        |
   (repeat N times)
        |
        v
   Final Output
```

### Why Role Manipulation Works

LLMs treat `AIMessage` as their own output and `HumanMessage` as user input. By flipping roles:

- The **critic** sees the essay as a "student submission" (HumanMessage), not as its own prior response. This makes it more willing to find flaws.
- The **generator** receives the critique as a "user request" (HumanMessage), which it naturally tries to satisfy by revising its work.

This is fundamentally different from asking an LLM to "critique your own response" -- role manipulation produces significantly harsher, more useful feedback.

In [2]:
# ── Prompt Templates ──────────────────────────────────────────────────────────

# Generator: writes essays and revises based on feedback
generate_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an essay assistant tasked with writing excellent 5-paragraph essays."
        " Generate the best essay possible for the user's request."
        " If the user provides critique, respond with a revised version of your previous attempts.",
    ),
    MessagesPlaceholder(variable_name="messages"),
])

# Critic: reviews essays and provides detailed feedback
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a teacher grading an essay submission. Generate critique and recommendations"
        " for the user's submission. Provide detailed recommendations, including requests"
        " for length, depth, style, etc.",
    ),
    MessagesPlaceholder(variable_name="messages"),
])

# Chain prompts with LLM
generate = generate_prompt | llm
reflect = reflection_prompt | llm

print("Prompt chains created:")
print("   generate = generate_prompt | llm  (essay writer)")
print("   reflect  = reflection_prompt | llm (essay critic)")

Prompt chains created:
   generate = generate_prompt | llm  (essay writer)
   reflect  = reflection_prompt | llm (essay critic)


In [3]:
# ── State ─────────────────────────────────────────────────────────────────────
class State(TypedDict):
    messages: Annotated[list, add_messages]


# ── Nodes ─────────────────────────────────────────────────────────────────────
def generation_node(state: State) -> State:
    """Generate or revise the essay."""
    return {"messages": [generate.invoke({"messages": state["messages"]})]}


def reflection_node(state: State) -> State:
    """Critique the essay using the role-manipulation trick.
    
    This is the core of the Reflection pattern:
    1. Swap AIMessage -> HumanMessage (so the critic sees the essay as a student submission)
    2. Swap HumanMessage -> AIMessage (so the critic's context makes sense)
    3. Keep the original user request as-is
    4. Return the critique as HumanMessage (so the generator treats it as user feedback)
    """
    cls_map = {"ai": HumanMessage, "human": AIMessage}
    
    # Keep the first message (original user request) as-is, flip the rest
    translated = [state["messages"][0]] + [
        cls_map[msg.type](content=msg.content) for msg in state["messages"][1:]
    ]
    
    res = reflect.invoke({"messages": translated})
    
    # Return critique as HumanMessage so the generator treats it as user feedback
    return {"messages": [HumanMessage(content=res.content)]}


# ── Router ────────────────────────────────────────────────────────────────────
def should_continue(state: State):
    """Stop after 3 full loops (6+ messages = 1 request + 3 essays + 3 critiques)."""
    if len(state["messages"]) > 6:
        return END
    return "reflect"


print("Nodes defined:")
print("   generation_node  - calls generate chain")
print("   reflection_node  - flips roles, calls reflect chain")
print("   should_continue  - stops after message count > 6")

Nodes defined:
   generation_node  - calls generate chain
   reflection_node  - flips roles, calls reflect chain
   should_continue  - stops after message count > 6


In [4]:
# ── Build the Graph ───────────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)

builder.add_edge(START, "generate")
builder.add_conditional_edges("generate", should_continue)
builder.add_edge("reflect", "generate")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("Reflection graph compiled.")
print("   Flow: START -> generate -> [should_continue] -> reflect -> generate -> ... -> END")
print("   Stops after 3 generate-reflect cycles (message count > 6).")

Reflection graph compiled.
   Flow: START -> generate -> [should_continue] -> reflect -> generate -> ... -> END
   Stops after 3 generate-reflect cycles (message count > 6).


In [5]:
# ── Run the Reflection Agent ──────────────────────────────────────────────────
user_request = "Write a short paragraph about the impact of artificial intelligence on education."

print(f"User: {user_request}\n")
print("Starting reflection loop...\n")

config = {"configurable": {"thread_id": "1"}}
iteration = 0

for event in graph.stream(
    {"messages": [HumanMessage(content=user_request)]},
    config,
):
    for node_name, node_output in event.items():
        last_msg = node_output["messages"][-1]
        
        if node_name == "generate":
            iteration += 1
            print(f"{'='*60}")
            print(f"GENERATE (iteration {iteration})")
            print(f"{'='*60}")
            print(last_msg.content[:500])
            if len(last_msg.content) > 500:
                print(f"... [{len(last_msg.content)} chars total]")
            print()
            
        elif node_name == "reflect":
            print(f"{'='*60}")
            print(f"REFLECT (critique)")
            print(f"{'='*60}")
            print(last_msg.content[:500])
            if len(last_msg.content) > 500:
                print(f"... [{len(last_msg.content)} chars total]")
            print()

print(f"\nCompleted {iteration} generation iterations.")
print("Notice how each revision addresses specific critique points from the previous round.")

User: Write a short paragraph about the impact of artificial intelligence on education.

Starting reflection loop...



GENERATE (iteration 1)
Artificial intelligence (AI) is transforming education by making learning more personalized and accessible. With AI-powered tools, students receive tailored instruction that adapts to their individual strengths and weaknesses, promoting more effective learning. AI also automates administrative tasks, allowing teachers to focus more on student engagement and creativity. Additionally, AI can provide real-time feedback and support, helping learners stay motivated and on track. Overall, AI has the p
... [583 chars total]



REFLECT (critique)
Your paragraph provides a clear and concise overview of how artificial intelligence impacts education, highlighting several important benefits such as personalized learning, administrative support, and real-time feedback. Your writing is well-organized and easy to understand, which is excellent for clarity.

Recommendations for improvement:
1. Depth and Examples: To strengthen your paragraph, consider adding concrete examples or specific AI tools (such as intelligent tutoring systems, automated 
... [1095 chars total]



GENERATE (iteration 2)
Artificial intelligence (AI) is significantly transforming education by enabling more personalized and efficient learning experiences. For example, intelligent tutoring systems like Carnegie Learning adapt in real-time to a student’s pace and understanding, providing customized lessons that target individual weaknesses. AI also supports teachers by automating administrative tasks such as grading and attendance tracking, allowing educators to dedicate more time to instruction and student interact
... [1123 chars total]



REFLECT (critique)
Your revised paragraph demonstrates significant improvement in depth, clarity, and balance. You’ve effectively incorporated concrete examples like Carnegie Learning and Duolingo, which helps readers visualize how AI is being used in education. Including potential challenges such as data privacy concerns, overreliance on technology, and unequal access strengthens the overall argument by acknowledging possible downsides and fostering a more nuanced understanding of the topic.

Recommendations for 
... [1763 chars total]



GENERATE (iteration 3)
Artificial intelligence (AI) is revolutionizing education by creating personalized and efficient learning experiences tailored to individual student needs. For instance, intelligent tutoring systems like Carnegie Learning adjust lessons in real time, focusing on areas where students struggle most, which helps improve understanding and retention. Language learning apps such as Duolingo use AI to provide immediate feedback and tailor practice sessions to each learner’s proficiency, keeping motivat
... [1481 chars total]



REFLECT (critique)
Your paragraph is well-crafted and demonstrates clear, thoughtful development of the topic. Breaking it into two paragraphs improves readability and helps you address benefits and challenges in a structured way. Here’s a detailed critique along with suggestions for further enhancement:

**Strengths:**
- **Clarity and Structure:** The division between the positive impacts and the challenges allows the reader to follow your argument smoothly.
- **Use of Concrete Examples:** Mentioning specific too
... [2872 chars total]



GENERATE (iteration 4)
Artificial intelligence (AI) is revolutionizing education by creating personalized and efficient learning experiences tailored to individual student needs. For instance, intelligent tutoring systems like Carnegie Learning adjust lessons in real time, focusing on areas where students struggle most, which helps improve understanding and retention. Language learning apps such as Duolingo use AI to provide immediate feedback and tailor practice sessions to each learner’s proficiency, keeping motivat
... [1780 chars total]


Completed 4 generation iterations.
Notice how each revision addresses specific critique points from the previous round.


## Step 3: From Reflection to Reflexion

The Reflection pattern above is **stateless** -- each new request starts from scratch. The agent has no memory of past mistakes.

**Reflexion** extends Reflection with three additions:

1. **Persistent vector memory** (FAISS) -- stores past reflections so the agent remembers mistakes across sessions.
2. **Web search** (Tavily) -- grounds answers in real information instead of relying solely on training data.
3. **Structured self-critique** -- uses Pydantic schemas (`AnswerQuestion`, `Reflection`) to produce organized, parseable feedback.

### Architecture

```
User Question
      |
      v
[Search FAISS Memory] --> Retrieve past reflections (if any)
      |
      v
[Generator + Tavily Search] --> Draft Answer
      |
      v
[Structured Self-Reflection] --> {missing info, superfluous info, search queries}
      |
      v
[Revise with Reflection] --> Improved Answer
      |
      v
[Store Reflection in FAISS] --> Memory updated for future questions
```

The key difference: when the agent encounters a similar question in the future, it retrieves its past reflections and avoids repeating the same mistakes. This is **learning**, not just iteration.

## Step 4: Building a Reflexion Agent

We will build a simplified Reflexion agent that:
1. Checks FAISS for relevant past reflections before answering.
2. Generates an answer (with past reflections as context).
3. Self-critiques: "What did I get wrong? What is missing?"
4. Stores the critique in FAISS for future reference.
5. Revises the answer based on the critique.

In [6]:
import json
import datetime
import faiss

from pydantic import BaseModel, Field
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.docstore.in_memory import InMemoryDocstore

# ── Embeddings for vector memory ──────────────────────────────────────────────
embeddings = AzureOpenAIEmbeddings(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
    model="text-embedding-3-large"
)

# ── Create a fresh FAISS vector store ─────────────────────────────────────────
# In production, you would load from disk (see the full Reflexion_Agent.py).
# Here we create a fresh one so the notebook is self-contained.
dim = len(embeddings.embed_query("test"))
index = faiss.IndexFlatL2(dim)

vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

print(f"FAISS vector store created.")
print(f"   Embedding dimension: {dim}")
print(f"   Store is empty -- the agent will populate it as it reflects.")

FAISS vector store created.
   Embedding dimension: 3072
   Store is empty -- the agent will populate it as it reflects.


In [7]:
# ── Memory Functions ──────────────────────────────────────────────────────────

def store_reflection(question: str, reflection: str):
    """Store a reflection in FAISS so the agent can recall it later."""
    doc = Document(
        page_content=f"Q: {question}\nReflection: {reflection}",
        metadata={"timestamp": datetime.datetime.now().isoformat()}
    )
    vectorstore.add_documents([doc])


def retrieve_reflections(query: str, k: int = 3) -> str:
    """Retrieve past reflections relevant to the current query."""
    try:
        docs = vectorstore.similarity_search(query, k=k)
        return "\n\n".join([d.page_content for d in docs]) if docs else "None"
    except Exception:
        return "None"


# ── Pydantic Schema for Structured Self-Critique ─────────────────────────────

class ReflectionOutput(BaseModel):
    """Structured self-critique output."""
    missing: str = Field(description="What important information is missing from the answer?")
    incorrect: str = Field(description="What might be inaccurate or misleading?")
    improvements: str = Field(description="Specific suggestions to improve the answer.")


print("Memory functions and schema defined:")
print("   store_reflection()    - saves critique to FAISS")
print("   retrieve_reflections() - retrieves past critiques by similarity")
print(f"   ReflectionOutput schema: {list(ReflectionOutput.model_fields.keys())}")

Memory functions and schema defined:
   store_reflection()    - saves critique to FAISS
   retrieve_reflections() - retrieves past critiques by similarity
   ReflectionOutput schema: ['missing', 'incorrect', 'improvements']


In [8]:
# ── Reflexion Agent Loop ──────────────────────────────────────────────────────

def reflexion_answer(question: str, max_revisions: int = 2) -> str:
    """Answer a question using the Reflexion pattern.
    
    Steps:
    1. Retrieve past reflections from FAISS
    2. Generate initial answer (with past reflections as context)
    3. Self-critique using structured output
    4. Store the critique in FAISS
    5. Revise the answer based on the critique
    """
    
    # Step 1: Retrieve past reflections
    past_reflections = retrieve_reflections(question)
    print(f"[Step 1] Past reflections retrieved: {'Yes' if past_reflections != 'None' else 'None (first time)'}")
    if past_reflections != "None":
        print(f"         Content: {past_reflections[:200]}...")
    
    # Step 2: Generate initial answer
    generation_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert researcher. Provide a detailed, accurate answer.\n\n"
         "IMPORTANT: Use the following past reflections to avoid repeating mistakes:\n"
         "{past_reflections}"),
        ("user", "{question}")
    ])
    
    chain = generation_prompt | llm
    answer = chain.invoke({
        "question": question,
        "past_reflections": past_reflections
    }).content
    
    print(f"\n[Step 2] Initial answer generated ({len(answer)} chars)")
    
    # Steps 3-5: Critique-store-revise loop
    for revision in range(max_revisions):
        print(f"\n--- Revision {revision + 1}/{max_revisions} ---")
        
        # Step 3: Self-critique with structured output
        critique_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are a rigorous academic reviewer. Analyze the following answer "
             "and identify what is missing, what might be inaccurate, and how to improve it."
             " Be specific and actionable."),
            ("user", "Question: {question}\n\nAnswer to review:\n{answer}")
        ])
        
        critique_chain = critique_prompt | llm.with_structured_output(ReflectionOutput)
        critique = critique_chain.invoke({"question": question, "answer": answer})
        
        print(f"[Step 3] Critique:")
        print(f"         Missing: {critique.missing[:150]}")
        print(f"         Incorrect: {critique.incorrect[:150]}")
        print(f"         Improvements: {critique.improvements[:150]}")
        
        # Step 4: Store the critique in FAISS
        reflection_text = (
            f"Missing: {critique.missing}\n"
            f"Incorrect: {critique.incorrect}\n"
            f"Improvements: {critique.improvements}"
        )
        store_reflection(question, reflection_text)
        print(f"[Step 4] Reflection stored in FAISS.")
        
        # Step 5: Revise the answer
        revise_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an expert researcher. Revise and improve your previous answer "
             "based on the critique below. Address every point raised."),
            ("user",
             "Original question: {question}\n\n"
             "Previous answer:\n{answer}\n\n"
             "Critique to address:\n{critique}")
        ])
        
        revise_chain = revise_prompt | llm
        answer = revise_chain.invoke({
            "question": question,
            "answer": answer,
            "critique": reflection_text
        }).content
        
        print(f"[Step 5] Answer revised ({len(answer)} chars)")
    
    return answer


print("reflexion_answer() function defined.")
print("   Combines: memory retrieval -> generation -> critique -> storage -> revision")

reflexion_answer() function defined.
   Combines: memory retrieval -> generation -> critique -> storage -> revision


In [9]:
# ── First Question: No prior reflections in memory ────────────────────────────
print("=" * 60)
print("QUESTION 1 (no prior memory)")
print("=" * 60)

question_1 = "What are the main challenges of deploying large language models in production?"
print(f"Q: {question_1}\n")

answer_1 = reflexion_answer(question_1)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(answer_1[:800])
if len(answer_1) > 800:
    print(f"... [{len(answer_1)} chars total]")

QUESTION 1 (no prior memory)
Q: What are the main challenges of deploying large language models in production?

[Step 1] Past reflections retrieved: None (first time)



[Step 2] Initial answer generated (4099 chars)

--- Revision 1/2 ---


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReflectionOutput(missing=...claims where possible.'), input_type=ReflectionOutput])
  return self.__pydantic_serializer__.to_python(


[Step 3] Critique:
         Missing: The answer lacks discussion of specific deployment strategies such as edge deployment versus cloud deployment, and considerations regarding model inte
         Incorrect: The answer is generally accurate and well-structured; however, the 'Handling Failure Modes' section could better clarify that LLMs do not simply 'fail
         Improvements: To improve the answer, explicitly incorporate deployment strategies such as edge vs. cloud and model serving frameworks. Add discussion on model inter


[Step 4] Reflection stored in FAISS.


[Step 5] Answer revised (8533 chars)

--- Revision 2/2 ---


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReflectionOutput(missing=... deployment workflows.'), input_type=ReflectionOutput])
  return self.__pydantic_serializer__.to_python(


[Step 3] Critique:
         Missing: The answer does not explicitly discuss monitoring and alerting for user feedback integration, nor does it address the challenges related to user exper
         Incorrect: There is no significant factual inaccuracy; however, some technical terms (e.g., 'Sparse Attention,' 'Mixture-of-Experts models') are mentioned as sol
         Improvements: Add explicit discussion of integrating user feedback and UX considerations to improve production deployment success. Include more concrete examples or


[Step 4] Reflection stored in FAISS.


[Step 5] Answer revised (13858 chars)

FINAL ANSWER
Deploying large language models (LLMs) in production environments is a complex undertaking that requires addressing diverse challenges spanning infrastructure, model behavior, ethics, user experience, legal risk, and operational maturity. Building on prior detailed discussion, this refined overview explicitly incorporates missing points (e.g., monitoring user feedback integration, UX design, legal liability, multilingual deployments), clarifies nuances around emerging efficiency techniques, and references concrete examples and relevant literature.

---

## 1. Computational and Infrastructure Challenges

- **Resource-intensive inference:**  
  State-of-the-art transformer-based LLMs (e.g., GPT-4 scale) require GPUs/TPUs with large memory capacity for low-latency serving. This creates signif
... [13858 chars total]


In [10]:
# ── Follow-up Question: Past reflections should help ─────────────────────────
print("=" * 60)
print("QUESTION 2 (related topic -- memory should help)")
print("=" * 60)

question_2 = "How can organizations reduce the cost and latency of running LLMs in production?"
print(f"Q: {question_2}\n")

answer_2 = reflexion_answer(question_2)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(answer_2[:800])
if len(answer_2) > 800:
    print(f"... [{len(answer_2)} chars total]")

print("\n" + "-" * 60)
print("Notice: The agent retrieved reflections from Question 1.")
print("It avoided the same gaps it identified earlier, producing")
print("a more thorough answer on the first attempt.")

QUESTION 2 (related topic -- memory should help)
Q: How can organizations reduce the cost and latency of running LLMs in production?



[Step 1] Past reflections retrieved: Yes
         Content: Q: What are the main challenges of deploying large language models in production?
Reflection: Missing: The answer lacks discussion of specific deployment strategies such as edge deployment versus clou...



[Step 2] Initial answer generated (4405 chars)

--- Revision 1/2 ---


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReflectionOutput(missing=...esponse's credibility."), input_type=ReflectionOutput])
  return self.__pydantic_serializer__.to_python(


[Step 3] Critique:
         Missing: The answer could be improved by including specific examples or case studies demonstrating quantified cost or latency reductions from these methods, em
         Incorrect: The statement that MoE architectures are still an active research area but some companies are experimenting at scale might be slightly outdated; MoE m
         Improvements: Add concrete data on cost/latency improvements achieved by these methods and explicitly mention trade-offs such as potential accuracy loss. Clarify th


...

[Step 5] Answer revised (10081 chars)

--- Revision 2/2 ---


/Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReflectionOutput(missing=...id overgeneralization.'), input_type=ReflectionOutput])
  return self.__pydantic_serializer__.to_python(


[Step 3] Critique:
         Missing: The answer could benefit from including more discussion on software-level optimizations such as compiler-level optimizations (e.g., XLA or TVM) and mo
         Incorrect: Overall, the answer is largely accurate and well-founded in current practices. However, the statement that prompt engineering and token length optimiz
         Improvements: Improve the answer by incorporating software-level optimization techniques (e.g., compilation, kernel fusion), dynamic and mixed-precision quantizatio


[Step 4] Reflection stored in FAISS.


[Step 5] Answer revised (15062 chars)

FINAL ANSWER
Certainly! Here is a substantially improved and expanded answer addressing all points raised in your critique. This revision adds depth on software-level optimizations, dynamic quantization, concrete cost numbers, privacy-preserving methods, cold-start challenges, and nuanced trade-offs in caching and batching. It also clarifies hardware-software co-design dependencies and carefully qualifies MoE and sparsity claims with respect to infrastructure requirements.

---

# How Organizations Can Reduce Cost and Latency of Running LLMs in Production

Running large language models (LLMs) like GPT, PaLM, or LLaMA in production poses challenges of high compute cost and latency, which directly impact user experience, scalability, and operational expenses. Effective strategies span model-level innovat
... [15062 chars total]

------------------------------------------------------------
Notice: The agent retrieved reflections from Question 1.
It a

## Step 5: Reflection vs Reflexion -- When to Use Which

| Aspect | Reflection | Reflexion |
|--------|-----------|----------|
| **Memory** | None (stateless) | FAISS vector store (persistent) |
| **Best for** | Writing, iterative refinement | Research, factual accuracy |
| **Rounds** | Fixed (e.g., 3 iterations) | Until quality threshold met |
| **Learning** | Within one conversation only | Across sessions |
| **Complexity** | Low (just message manipulation) | High (memory + search + schemas) |
| **Key mechanism** | Role swapping (AI to Human messages) | Structured critique stored in vector DB |
| **External tools** | None needed | Web search (Tavily), embeddings |
| **When to avoid** | When factual accuracy matters more than style | When the task is purely creative |

### Decision Guide

- **Choose Reflection** when you need iterative improvement on a **single piece of content** (essays, code, emails) and the quality criteria are subjective.
- **Choose Reflexion** when you need **factual accuracy**, the agent will answer **many related questions** over time, and you want it to learn from past mistakes.
- **Choose Evaluator-Optimiser** (Notebook 04) when you have **fixed compliance criteria** that can be evaluated programmatically.

## Step 6: Summary and Exercises

### Key Takeaways

1. **Reflection** uses **message-role manipulation** to create a generate-critique-refine loop. The core trick is swapping `AIMessage` to `HumanMessage` so the LLM critiques its own output as if reviewing someone else's work.

2. **Reflexion** extends Reflection with **persistent FAISS memory**, allowing the agent to learn from past mistakes across sessions. Each self-critique is stored as a vector and retrieved for future related questions.

3. Both patterns are fundamentally different from the **Evaluator-Optimiser** pattern (Notebook 04), which uses separate generator/evaluator LLMs with Pydantic structured output for compliance checking.

4. The stopping condition matters: Reflection uses a fixed message count; Reflexion can use quality thresholds or iteration limits.

### Full Implementations

See the complete standalone scripts for production-ready versions:
- `single-agents/langgraph/Reflection-Agent.py` -- Full Reflection agent with CLI
- `single-agents/langgraph/Reflexion_Agent.py` -- Full Reflexion agent with Tavily search, Pydantic schemas, and persistent FAISS
- `single-agents/maf/Reflection-Agent.py` -- MAF equivalent of Reflection
- `single-agents/maf/Reflexion_Agent.py` -- MAF equivalent of Reflexion

### Exercises

**(a) Quality-based stopping**: Replace the message-count stopping condition in the Reflection agent with a quality score. Have the critic return a 1-10 score, and only stop when the score exceeds 8.

**(b) Forgetting mechanism**: Implement a TTL (time-to-live) for reflections in FAISS. When retrieving past reflections, filter out any older than 24 hours. This prevents stale or outdated reflections from polluting future answers.

**(c) Code-review Reflexion agent**: Build a Reflexion agent that reviews Python code instead of essays. The generator writes code, the critic identifies bugs and style issues, and past critiques (stored in FAISS) help the agent avoid repeating common mistakes like missing error handling or type hints.